# 📘 Capítulo 4 – Avaliação e Validação Temporal
**Autor:** Rodrigo Santana Ferreira  
**Disciplina:** Séries Temporais  

---
Este notebook contém os scripts Python apresentados no Capítulo 4, organizados por seção conforme o material da aula.

## 🔧 Instalação de Dependências

In [1]:
# Instalação das bibliotecas necessárias
!pip install pandas numpy scikit-learn matplotlib

## Passo 1 – Preparação dos Dados e Criação de Features
Carregamento do dataset AirPassengers e criação das features temporais básicas (seção HANDS ON).

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [3]:
# Carregar dataset clássico
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')

In [4]:
# Converte a coluna para frequência mensal
df.columns = ['Passengers']
df = df.asfreq('MS')

In [5]:
# imprime o dataframe
df.head()

,Passengers
Month,
1949-01-01,112
1949-02-01,118
1949-03-01,132
1949-04-01,129
1949-05-01,121



### Criar features temporais básicas
---



In [6]:
df['lag_1'] = df['Passengers'].shift(1)
df['lag_12'] = df['Passengers'].shift(12)
df['diff_1'] = df['Passengers'].diff(1)

In [7]:
df['rolling_mean_3'] = df['Passengers'].rolling(3).mean()
df['month'] = df.index.month
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

In [8]:
# imprime o dataframe
df.head()

,Passengers,lag_1,lag_12,diff_1,rolling_mean_3,month,sin_month,cos_month
Month,,,,,,,,
1949-01-01,112,NaN,NaN,NaN,NaN,1,0.500000,8.660254e-01
1949-02-01,118,112.0,NaN,6.0,NaN,2,0.866025,5.000000e-01
1949-03-01,132,118.0,NaN,14.0,120.666667,3,1.000000,6.123234e-17
1949-04-01,129,132.0,NaN,-3.0,126.333333,4,0.866025,-5.000000e-01
1949-05-01,121,129.0,NaN,-8.0,127.333333,5,0.500000,-8.660254e-01


In [9]:
# Separa X e Y
df = df.dropna()
X = df.drop('Passengers', axis=1)
y = df['Passengers']

In [10]:
# imprime o dataframe X
X.head()

,lag_1,lag_12,diff_1,rolling_mean_3,month,sin_month,cos_month
Month,,,,,,,
1950-01-01,118.0,112.0,-3.0,112.333333,1,0.500000,8.660254e-01
1950-02-01,115.0,118.0,11.0,119.666667,2,0.866025,5.000000e-01
1950-03-01,126.0,132.0,15.0,127.333333,3,1.000000,6.123234e-17
1950-04-01,141.0,129.0,-6.0,134.000000,4,0.866025,-5.000000e-01
1950-05-01,135.0,121.0,-10.0,133.666667,5,0.500000,-8.660254e-01


In [11]:
# imprime o dataframe y
y.head()

,Passengers
Month,
1950-01-01,115
1950-02-01,126
1950-03-01,141
1950-04-01,135
1950-05-01,125


In [12]:
# imprime o dataframe
print(f"Shape final: {X.shape}")

Shape final: (132, 7)


## Passo 2 – Validação Errada: O que NUNCA Fazer
Demonstração do problema de data leakage com validação aleatória em séries temporais.

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
# Validação aleatória (errada para séries temporais)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Treina o modelo GBR
model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

In [16]:
# Previsões com o dataset de test
pred = model.predict(X_test)

In [17]:
print(f"MAE (validação incorreta): {mean_absolute_error(y_test, pred):.2f}")
print("→ Este resultado é ilusório! O modelo viu o futuro.")

MAE (validação incorreta): 8.76
→ Este resultado é ilusório! O modelo viu o futuro.


## Passo 3 – Validação Correta com TimeSeriesSplit
Avaliação respeitando a ordem temporal — sem vazamento de dados futuros.

In [18]:
tscv = TimeSeriesSplit(n_splits=5)
mae_scores, rmse_scores, mape_scores = [], [], []

In [19]:
# Avaliação com folds
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = GradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)

In [20]:
print(f"Fold {fold}: MAE = {mae:.2f} | RMSE = {rmse:.2f} | MAPE = {mape:.2f}%")

Fold 5: MAE = 37.69 | RMSE = 52.13 | MAPE = 7.53%


In [21]:
print(f"\nMédia TimeSeriesSplit → MAE: {np.mean(mae_scores):.2f} | RMSE: {np.mean(rmse_scores):.2f} | MAPE: {np.mean(mape_scores):.2f}%")


Média TimeSeriesSplit → MAE: 28.53 | RMSE: 38.08 | MAPE: 8.33%


## Passo 4 – Rolling Forecast (Walk-Forward Validation)
Simulação do uso real em produção: re-treino a cada janela de previsão.

In [22]:
def rolling_forecast_evaluation(X, y, test_size=12, n_windows=5):
    mae_list = []
    for i in range(n_windows):
        train_end = len(X) - (n_windows - i) * test_size
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]
        X_test = X.iloc[train_end:train_end + test_size]
        y_test = y.iloc[train_end:train_end + test_size]

        model = GradientBoostingRegressor(random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mae_list.append(mean_absolute_error(y_test, pred))
    return np.mean(mae_list)

In [23]:
print(f"MAE médio (Rolling Forecast): {rolling_forecast_evaluation(X, y):.2f}")

MAE médio (Rolling Forecast): 23.21


## Passo 5 – Comparação com Modelo Naïve (Baseline)
O modelo naïve é o benchmark mínimo: qualquer modelo sério deve superá-lo.

In [24]:
# Previsão naïve: valor do período anterior
naive_pred = y.shift(1).dropna()
y_naive = y.iloc[1:]

In [25]:
print(f"MAE Naïve: {mean_absolute_error(y_naive, naive_pred):.2f}")

MAE Naïve: 27.32
